# Cross-Position Attention

Analyze information flow between positions: position importance, flow matrices,
pairwise interactions, and bottleneck detection.

## Why This Matters

Attention cross position reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.cross_position_attention import (
    position_information_flow, source_position_importance,
    attention_flow_matrix, position_pair_interaction,
    attention_bottleneck_positions,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Position Information Flow

How much information flows to/from each position at a given layer?

In [ ]:
for layer in range(4):
    result = position_information_flow(model, tokens, layer=layer)
    print(f"Layer {layer} ({result['n_hubs']} hubs):")
    for p in result['per_position']:
        hub = ' [HUB]' if p['is_hub'] else ''
        print(f"  Pos {p['position']}: recv={p['total_received']:.3f}, "
              f"sent={p['total_sent']:.3f}{hub}")
    print()

## Source Position Importance

Which source positions matter most for the final prediction?

In [ ]:
result = source_position_importance(model, tokens, layer=2, target_position=-1)
print(f"Target: pos {result['target_position']}, Top source: pos {result['top_source']}\n")
for s in result['per_source']:
    bar = '█' * int(s['total_attention'] * 10)
    print(f"  Src pos {s['source_position']} (tok {s['source_token']:2d}): "
          f"attn={s['total_attention']:.3f} (max head {s['max_head']}: {s['max_head_attention']:.3f}) {bar}")

## Attention Flow Matrix

Full-model attention flow averaged across all layers and heads.

In [ ]:
result = attention_flow_matrix(model, tokens)
print(f"Mean flow: {result['mean_flow']:.4f}\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: total_incoming={p['total_incoming_flow']:.3f}, "
          f"primary_source=pos {p['primary_source']}")

## Position Pair Interaction

Detailed per-layer, per-head analysis of attention between two positions.

In [ ]:
result = position_pair_interaction(model, tokens, pos_a=1, pos_b=5)
print(f"Pos {result['pos_a']} → Pos {result['pos_b']}")
print(f"Total attention: {result['total_attention']:.4f}, "
      f"mean/layer: {result['mean_per_layer']:.4f}\n")
for layer_info in result['per_layer']:
    heads = ', '.join(f"H{h['head']}:{h['attention']:.3f}" for h in layer_info['per_head'])
    print(f"  L{layer_info['layer']}: total={layer_info['layer_total']:.3f}, "
          f"max=H{layer_info['max_head']} ({heads})")

## Attention Bottleneck Positions

Find positions that many later positions attend to heavily —
information bottlenecks in the sequence.

In [ ]:
result = attention_bottleneck_positions(model, tokens)
print(f"Bottlenecks: {result['n_bottlenecks']}, "
      f"top: pos {result['top_bottleneck']}\n")
for p in result['per_position']:
    bar = '█' * int(p['incoming_attention'] * 3)
    bn = ' [BOTTLENECK]' if p['is_bottleneck'] else ''
    print(f"  Pos {p['position']} (tok {p['token']:2d}): "
          f"incoming={p['incoming_attention']:.3f} {bar}{bn}")